<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">01_Demo_Required_Setup</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# Required Setup
### Duration - 2 Minutes

<p style="font-size: 1em; line-height: 1.6; color: #333">Run this notebook once before starting any demos or labs. It initializes your personal catalog, creates the <code>intro_spark</code> schema and the volumes used throughout the course, exports the customer CSV source files used in the reading and writing demo, and generates the synthetic flight data used in the ETL demo.</p>

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations explore the Spark UI and cluster internals, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## Step 1 of 4 - Catalog Setup

Detects your workspace type and sets your personal catalog as the default. Creates a `labuser_<yourname>` catalog if it does not already exist.

In [0]:
%run ./Includes/Classroom-Setup-Common

## Step 2 of 4 - Create the Course Schema and Volumes

Creates the `intro_spark` schema along with two volumes: `source_files` holds the CSV files the course reads from, and `output` is the destination for files you write during the demos.

In [0]:
-- Create Schema and Volumes
CREATE SCHEMA IF NOT EXISTS intro_spark;
USE SCHEMA intro_spark;

CREATE VOLUME IF NOT EXISTS source_files;
CREATE VOLUME IF NOT EXISTS output;

SELECT current_catalog() AS catalog, current_schema() AS schema;

## Step 3 of 4 - Export Customer CSV Source Files

Exports the TPC-H customer table from the `samples` catalog to a directory of CSV files in the `source_files` volume. The reading and writing demo uses these files to demonstrate schema inference and explicit schemas.

In [0]:
%python
# Export Customer CSV Source Files
catalog = spark.catalog.currentCatalog()

customers_src = spark.table("samples.tpch.customer").selectExpr(
    "c_custkey AS customer_id",
    "c_name AS name",
    "c_address AS address",
    "c_nationkey AS nation_key",
    "c_phone AS phone",
    "CAST(c_acctbal AS DOUBLE) AS account_balance",
    "c_mktsegment AS market_segment",
    "c_comment AS comment"
)

(customers_src
    .repartition(8)
    .write
    .mode("overwrite")
    .option("header", "true")
    .format("csv")
    .save(f"/Volumes/{catalog}/intro_spark/source_files/customers"))

print(f"Exported {customers_src.count()} customer records to /Volumes/{catalog}/intro_spark/source_files/customers")

## Step 4 of 4 - Generate Synthetic Flight Data

Generates one million synthetic flight records and saves them to the `flights` table. The data intentionally contains messy values - string columns holding numbers, `NA` markers, and nulls - because cleaning them is the point of the ETL demo in Section 4.

In [0]:
%python
# Generate Synthetic Flight Data
from pyspark.sql.functions import col, lit, when, rand, floor, lpad, concat, expr

rows = 1_000_000

flights = (
    spark.range(rows)
    .withColumn("Year", lit(2025))
    .withColumn("Month", (floor(rand(11) * 12) + 1).cast("int"))
    .withColumn("DayofMonth", (floor(rand(12) * 28) + 1).cast("int"))
    .withColumn("dep_hour", floor(rand(13) * 24).cast("int"))
    .withColumn("dep_min", floor(rand(14) * 60).cast("int"))
    .withColumn(
        "DepTime",
        when(rand(15) < 0.02, lit("NA")).otherwise(
            concat(lpad(col("dep_hour"), 2, "0"), lpad(col("dep_min"), 2, "0"))
        ),
    )
    .withColumn("FlightNum", (floor(rand(16) * 7000) + 1).cast("int"))
    .withColumn(
        "UniqueCarrier",
        expr("element_at(array('AA','DL','UA','WN','AS','B6'), CAST(floor(rand(17) * 6) + 1 AS INT))"),
    )
    .withColumn(
        "CRSElapsedTime",
        when(rand(18) < 0.01, None).otherwise((floor(rand(19) * 360) + 45).cast("int")),
    )
    .withColumn(
        "ActualElapsedTime",
        when(rand(20) < 0.02, lit("NA")).otherwise(
            (col("CRSElapsedTime") + floor(rand(21) * 60) - lit(20)).cast("int").cast("string")
        ),
    )
    .withColumn(
        "ArrDelay",
        when(rand(22) < 0.03, lit("NA")).otherwise(
            (floor(rand(23) * 120) - lit(30)).cast("int").cast("string")
        ),
    )
    .withColumn(
        "Origin",
        expr("element_at(array('SFO','LAX','JFK','ORD','ATL','SEA','DEN','BOS'), CAST(floor(rand(24) * 8) + 1 AS INT))"),
    )
    .withColumn(
        "Dest",
        expr("element_at(array('SFO','LAX','JFK','ORD','ATL','SEA','DEN','BOS'), CAST(floor(rand(25) * 8) + 1 AS INT))"),
    )
    .drop("id", "dep_hour", "dep_min")
)

flights.write.mode("overwrite").saveAsTable("flights")

print(f"Saved {spark.table('flights').count()} flight records to the flights table")

## Setup Complete

All catalogs, schemas, volumes, and source data are in place. You can now run any demo or lab notebook. Each demo and lab validates this setup automatically when its classroom setup runs.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>